In [2]:
# =============================================================================
# CELL 1 — Imports and environment verification
#
# Kernel: CFB Model (ARM)  ~/miniforge3/envs/cfb_model_arm/bin/python
# NumPyro 0.21.0 / JAX 0.10.0 confirmed working on cpu backend (Day 20).
#
# This is the full 4-chain production fit.
# model_04 was the diagnostic run (1 chain, 200 warmup, 200 samples).
# This notebook: 4 chains, 1000 warmup, 1000 samples.
#
# DB connection opened here and held open for the full notebook.
# Do not close until Cell 7 (artifact save) is complete.
# =============================================================================

import numpy as np
import pandas as pd
import psycopg2
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
from dataclasses import dataclass
from typing import Optional
import pickle
import time

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")
print(f"pandas          : {pd.__version__}")
print(f"numpy           : {np.__version__}")
print()
print("DB connection established.")
print("Cell 1 complete.")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu
pandas          : 3.0.2
numpy           : 2.4.4

DB connection established.
Cell 1 complete.


In [3]:
# =============================================================================
# CELL 2 — Conference index maps, GameData, and model_cfb()
#
# Copied verbatim from model_04 Cell 2 (audited and locked, Day 24).
# No changes to architecture, priors, or likelihood permitted here.
#
# Prior corrections applied (model_05 prior predictive audit, Day 24):
#   r_negbinom       : Gamma(16.0, 2.0)  mean=8, std=2
#   sigma_conference : HalfNormal(0.1)
#   sigma_attack     : HalfNormal(0.1)
#   sigma_defense    : HalfNormal(0.1)
#   sigma_hfa_team   : HalfNormal(0.1)
#   sp_weight        : Normal(0, 0.15)
#   b_off_archetype  : Normal(0, 0.15) x 4
#   b_def_archetype  : Normal(0, 0.15) x 4
#   all other game-level coefs : Normal(0, 0.15)
#   b_close_game_epa / b_close_game_def_epa : Normal(0, 0.10)
# =============================================================================

# --- Conference index maps ---
CONFERENCES = [
    "ACC",               # 0
    "American Athletic", # 1
    "Big 12",            # 2
    "Big Ten",           # 3
    "Conference USA",    # 4
    "Mid-American",      # 5
    "Mountain West",     # 6
    "Pac-12",            # 7
    "SEC",               # 8
    "Sun Belt",          # 9
]
N_CONFERENCES = len(CONFERENCES)
conf_to_idx   = {c: i for i, c in enumerate(CONFERENCES)}
SUN_BELT_IDX  = conf_to_idx["Sun Belt"]

# --- Conference-scope sets ---
SCOPE_LAST3_OFF_EPA      = {"ACC", "Mid-American", "SEC"}
SCOPE_LAST3_DEF_EPA      = {"American Athletic", "Big Ten", "Conference USA",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_LAST3_PTS_SCORED   = {"ACC", "Big 12", "Big Ten", "Conference USA",
                             "Mid-American", "Mountain West"}
SCOPE_LAST3_PTS_ALLOWED  = {"American Athletic", "Big Ten", "Conference USA",
                             "Mountain West", "Pac-12", "Sun Belt"}
SCOPE_DAYS_SINCE         = {"American Athletic", "Big 12"}
SCOPE_CLOSE_PLAY_COUNT   = {"ACC", "American Athletic", "Big 12",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_ELEVATION          = {"Mountain West", "Big 12"}


def build_conf_mask(team_conferences, scope_set):
    return jnp.array(
        [1.0 if c in scope_set else 0.0 for c in team_conferences],
        dtype=jnp.float32
    )


# --- GameData dataclass ---
@dataclass
class GameData:
    # Index arrays (int32)
    team_idx  : jnp.ndarray
    opp_idx   : jnp.ndarray
    conf_idx  : jnp.ndarray

    # Home indicator
    is_home   : jnp.ndarray

    # Observed scores
    points    : Optional[jnp.ndarray]

    # Prior seeds
    sp_rating          : jnp.ndarray
    recruiting_3yr_avg : jnp.ndarray

    # Universal game-level features
    close_game_epa        : jnp.ndarray
    close_game_def_epa    : jnp.ndarray
    pregame_elo           : jnp.ndarray
    elo_sp_divergence     : jnp.ndarray
    last3_win_pct         : jnp.ndarray
    away_travel_distance  : jnp.ndarray
    away_tz_delta         : jnp.ndarray
    wind_chill            : jnp.ndarray
    rush_rate_std_downs   : jnp.ndarray
    rush_rate_pass_downs  : jnp.ndarray
    off_sack_rate_allowed : jnp.ndarray
    def_sack_rate         : jnp.ndarray

    # Archetype index arrays — int32, values 0-3
    off_archetype_idx : jnp.ndarray
    def_archetype_idx : jnp.ndarray

    # Conference-scoped features + masks (7 pairs)
    last3_off_epa_avg      : jnp.ndarray
    mask_last3_off_epa     : jnp.ndarray
    last3_def_epa_avg      : jnp.ndarray
    mask_last3_def_epa     : jnp.ndarray
    last3_pts_scored_avg   : jnp.ndarray
    mask_last3_pts_scored  : jnp.ndarray
    last3_pts_allowed_avg  : jnp.ndarray
    mask_last3_pts_allowed : jnp.ndarray
    days_since_last_game   : jnp.ndarray
    mask_days_since        : jnp.ndarray
    close_play_count_delta : jnp.ndarray
    mask_close_play_count  : jnp.ndarray
    away_elevation_delta   : jnp.ndarray
    mask_elevation         : jnp.ndarray


# --- model_cfb() ---
def model_cfb(data: GameData, N_teams: int):

    # ── League level ──────────────────────────────────────────────────────────
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(16.0, 2.0),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Conference level ──────────────────────────────────────────────────────
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Team level — non-centered ─────────────────────────────────────────────
    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.1))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.1))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    # ── Prior seeds ───────────────────────────────────────────────────────────
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))

    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5), sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # ── Game-level coefficients ───────────────────────────────────────────────
    b_close_game_epa      = numpyro.sample("b_close_game_epa",      dist.Normal(0.0, 0.10))
    b_close_game_def_epa  = numpyro.sample("b_close_game_def_epa",  dist.Normal(0.0, 0.10))
    b_pregame_elo         = numpyro.sample("b_pregame_elo",         dist.Normal(0.0, 0.15))
    b_elo_sp_divergence   = numpyro.sample("b_elo_sp_divergence",   dist.Normal(0.0, 0.15))
    b_last3_win_pct       = numpyro.sample("b_last3_win_pct",       dist.Normal(0.0, 0.15))
    b_away_travel_distance= numpyro.sample("b_away_travel_distance",dist.Normal(0.0, 0.15))
    b_away_tz_delta       = numpyro.sample("b_away_tz_delta",       dist.Normal(0.0, 0.15))
    b_wind_chill          = numpyro.sample("b_wind_chill",          dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs = numpyro.sample("b_rush_rate_std_downs", dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs= numpyro.sample("b_rush_rate_pass_downs",dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed=numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate       = numpyro.sample("b_def_sack_rate",       dist.Normal(0.0, 0.15))

    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )

    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))

    # ── Linear predictor ──────────────────────────────────────────────────────
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight           * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa      * data.close_game_epa
        + b_close_game_def_epa  * data.close_game_def_epa
        + b_pregame_elo         * data.pregame_elo
        + b_elo_sp_divergence   * data.elo_sp_divergence
        + b_last3_win_pct       * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )

    # ── Likelihood ────────────────────────────────────────────────────────────
    mu = jnp.exp(log_mu).clip(1e-6)
    r  = r_negbinom[data.conf_idx].clip(1e-6)

    numpyro.sample(
        "obs",
        dist.NegativeBinomial2(mu, r, validate_args=False),
        obs=data.points
    )


print("Conference index map:")
for name, idx in conf_to_idx.items():
    tag = "  <- Sun Belt (rec_weight non-positive)" if idx == SUN_BELT_IDX else ""
    print(f"  {idx:2d} : {name}{tag}")
print(f"\nN_CONFERENCES : {N_CONFERENCES}")
print(f"SUN_BELT_IDX  : {SUN_BELT_IDX}")
print()
print("build_conf_mask() defined.")
print("GameData dataclass defined.")
print("model_cfb() defined.")
print("  r_negbinom       : Gamma(16.0, 2.0) x N_CONFERENCES  [mean=8, std=2]")
print("  sigma_conference : HalfNormal(0.1)")
print("  sigma_attack     : HalfNormal(0.1)")
print("  sigma_defense    : HalfNormal(0.1)")
print("  sigma_hfa_team   : HalfNormal(0.1)")
print("  sp_weight        : Normal(0.0, 0.15)")
print("  b_off_archetype  : Normal(0.0, 0.15) x 4")
print("  b_def_archetype  : Normal(0.0, 0.15) x 4")
print("  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])")
print("Cell 2 complete.")

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt  <- Sun Belt (rec_weight non-positive)

N_CONFERENCES : 10
SUN_BELT_IDX  : 9

build_conf_mask() defined.
GameData dataclass defined.
model_cfb() defined.
  r_negbinom       : Gamma(16.0, 2.0) x N_CONFERENCES  [mean=8, std=2]
  sigma_conference : HalfNormal(0.1)
  sigma_attack     : HalfNormal(0.1)
  sigma_defense    : HalfNormal(0.1)
  sigma_hfa_team   : HalfNormal(0.1)
  sp_weight        : Normal(0.0, 0.15)
  b_off_archetype  : Normal(0.0, 0.15) x 4
  b_def_archetype  : Normal(0.0, 0.15) x 4
  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])
Cell 2 complete.


In [4]:
# =============================================================================
# CELL 3 — Load training data from int.int_game_model_features
#
# Season filter: IN (2022, 2023, 2024). 2025 is holdout — never touched.
# FBS integrity check mandatory after load.
# All Decimal columns cast to float64 immediately.
# =============================================================================

load_sql = """
    SELECT *
    FROM int.int_game_model_features
    WHERE season IN (2022, 2023, 2024)
    ORDER BY season, game_id, team_name
"""

cur.execute(load_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

numeric_cols = [c for c in df.columns if c not in
                ['team_name', 'opponent', 'conference']]
df[numeric_cols] = df[numeric_cols].astype(float)

# --- FBS integrity check ---
print("Conference distribution:")
print(df['conference'].value_counts().sort_index().to_string())
print()

assert 'FBS Independents' not in df['conference'].values, \
    "FBS Independents found in conference — fix before proceeding"
assert 2025 not in df['season'].values, \
    "2025 holdout found in training data — fix before proceeding"
assert df['game_id'].isna().sum() == 0, \
    "Null game_id found"

# --- Shape and basic sanity ---
print(f"Rows loaded        : {len(df):,}")
print(f"Columns            : {len(df.columns)}")
print(f"Unique games       : {df['game_id'].nunique():,}")
print(f"Unique teams       : {df['team_name'].nunique():,}")
print(f"Seasons            : {sorted(df['season'].unique().tolist())}")
print(f"Nulls anywhere     : {df.isna().sum().sum()}")
print(f"points_scored mean : {df['points_scored'].mean():.1f}")
print()

assert len(df) == 3214,          f"Expected 3,214 rows, got {len(df):,}"
assert df['game_id'].nunique() == 1607, \
    f"Expected 1,607 games, got {df['game_id'].nunique():,}"
assert df.isna().sum().sum() == 0, "Nulls found in training data"

print("All load checks passed.")
print("Cell 3 complete.")

Conference distribution:
conference
ACC                  364
American Athletic    320
Big 12               366
Big Ten              420
Conference USA       246
Mid-American         294
Mountain West        282
Pac-12               222
SEC                  358
Sun Belt             342

Rows loaded        : 3,214
Columns            : 31
Unique games       : 1,607
Unique teams       : 131
Seasons            : [2022.0, 2023.0, 2024.0]
Nulls anywhere     : 0
points_scored mean : 26.9

All load checks passed.
Cell 3 complete.


In [5]:
# =============================================================================
# CELL 4 — Build index arrays and conference masks
#
# Copied verbatim from model_04 Cell 4 (audited and locked, Day 24).
# team_idx and opp_idx: 0-based over 131 unique teams, pooled across seasons.
# conf_idx: maps conference string to conf_to_idx integer from Cell 2.
# All masks built from team_conferences list (row-level conference strings).
# =============================================================================

# --- Team index map ---
unique_teams = sorted(df['team_name'].unique())
N_teams      = len(unique_teams)
team_to_idx  = {name: i for i, name in enumerate(unique_teams)}

print(f"N_teams : {N_teams}")

# --- Verify all opponents are in the team map ---
unknown_opps = set(df['opponent'].unique()) - set(team_to_idx.keys())
assert len(unknown_opps) == 0, \
    f"Opponents not in team_to_idx: {unknown_opps}"

# --- Build index arrays ---
team_idx = jnp.array(df['team_name'].map(team_to_idx).values, dtype=jnp.int32)
opp_idx  = jnp.array(df['opponent'].map(team_to_idx).values,  dtype=jnp.int32)
conf_idx = jnp.array(df['conference'].map(conf_to_idx).values, dtype=jnp.int32)

# --- Verify conference mapping was complete ---
assert df['conference'].map(conf_to_idx).isna().sum() == 0, \
    f"Unmapped conferences: {set(df['conference'].unique()) - set(conf_to_idx.keys())}"

# --- Range assertions ---
assert int(team_idx.min()) >= 0 and int(team_idx.max()) < N_teams, \
    f"team_idx out of range: [{int(team_idx.min())}, {int(team_idx.max())}]"
assert int(opp_idx.min()) >= 0 and int(opp_idx.max()) < N_teams, \
    f"opp_idx out of range: [{int(opp_idx.min())}, {int(opp_idx.max())}]"
assert int(conf_idx.min()) >= 0 and int(conf_idx.max()) < N_CONFERENCES, \
    f"conf_idx out of range: [{int(conf_idx.min())}, {int(conf_idx.max())}]"

print(f"team_idx : shape={team_idx.shape}  min={int(team_idx.min())}  max={int(team_idx.max())}")
print(f"opp_idx  : shape={opp_idx.shape}  min={int(opp_idx.min())}  max={int(opp_idx.max())}")
print(f"conf_idx : shape={conf_idx.shape}  min={int(conf_idx.min())}  max={int(conf_idx.max())}")

# --- Build conference masks ---
team_conferences = df['conference'].tolist()

mask_last3_off_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_OFF_EPA)
mask_last3_def_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_DEF_EPA)
mask_last3_pts_scored  = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_SCORED)
mask_last3_pts_allowed = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_ALLOWED)
mask_days_since        = build_conf_mask(team_conferences, SCOPE_DAYS_SINCE)
mask_close_play_count  = build_conf_mask(team_conferences, SCOPE_CLOSE_PLAY_COUNT)
mask_elevation         = build_conf_mask(team_conferences, SCOPE_ELEVATION)

print()
print("Conference mask non-zero counts (of 3,214 rows):")
print(f"  mask_last3_off_epa     : {int(mask_last3_off_epa.sum()):,}")
print(f"  mask_last3_def_epa     : {int(mask_last3_def_epa.sum()):,}")
print(f"  mask_last3_pts_scored  : {int(mask_last3_pts_scored.sum()):,}")
print(f"  mask_last3_pts_allowed : {int(mask_last3_pts_allowed.sum()):,}")
print(f"  mask_days_since        : {int(mask_days_since.sum()):,}")
print(f"  mask_close_play_count  : {int(mask_close_play_count.sum()):,}")
print(f"  mask_elevation         : {int(mask_elevation.sum()):,}")

assert int(mask_last3_off_epa.sum()) == 364 + 294 + 358, \
    f"mask_last3_off_epa sum wrong: {int(mask_last3_off_epa.sum())}"

print()
print("All index and mask checks passed.")
print("Cell 4 complete.")

N_teams : 131
team_idx : shape=(3214,)  min=0  max=130
opp_idx  : shape=(3214,)  min=0  max=130
conf_idx : shape=(3214,)  min=0  max=9

Conference mask non-zero counts (of 3,214 rows):
  mask_last3_off_epa     : 1,016
  mask_last3_def_epa     : 1,844
  mask_last3_pts_scored  : 1,972
  mask_last3_pts_allowed : 1,832
  mask_days_since        : 686
  mask_close_play_count  : 1,908
  mask_elevation         : 648

All index and mask checks passed.
Cell 4 complete.


In [6]:
# =============================================================================
# CELL 5 — Construct GameData
#
# Copied verbatim from model_04 Cell 5 (audited and locked, Day 24).
# Data loaded from int.int_game_model_features is already fully preprocessed
# by model_03 Cell 7. DO NOT re-standardize anything here.
#
# Column name reference (exact names as they exist in the table):
#   close_game_epa_per_play       -> close_game_epa
#   close_game_def_epa_per_play   -> close_game_def_epa
#   away_travel_distance_mi       -> away_travel_distance
#   away_tz_delta_hrs             -> away_tz_delta
#   rush_rate_std_downs_delta     -> rush_rate_std_downs
#   rush_rate_pass_downs_delta    -> rush_rate_pass_downs
#   off_sack_rate_allowed_delta   -> off_sack_rate_allowed
#   def_sack_rate_delta           -> def_sack_rate
#   last3_points_scored_avg       -> last3_pts_scored_avg
#   last3_points_allowed_avg      -> last3_pts_allowed_avg
#   close_game_play_count_delta   -> close_play_count_delta
#   away_elevation_delta_ft       -> away_elevation_delta
# =============================================================================

def to_f32(col):
    return jnp.array(df[col].values, dtype=jnp.float32)

def to_i32(col):
    return jnp.array(df[col].values, dtype=jnp.int32)

training_data = GameData(
    team_idx  = team_idx,
    opp_idx   = opp_idx,
    conf_idx  = conf_idx,

    is_home   = jnp.array(df['is_home'].values, dtype=jnp.float32),
    points    = jnp.array(df['points_scored'].values, dtype=jnp.int32),

    sp_rating          = to_f32('sp_rating'),
    recruiting_3yr_avg = to_f32('recruiting_3yr_avg'),

    close_game_epa        = to_f32('close_game_epa_per_play'),
    close_game_def_epa    = to_f32('close_game_def_epa_per_play'),
    pregame_elo           = to_f32('pregame_elo'),
    elo_sp_divergence     = to_f32('elo_sp_divergence'),
    last3_win_pct         = to_f32('last3_win_pct'),
    away_travel_distance  = to_f32('away_travel_distance_mi'),
    away_tz_delta         = to_f32('away_tz_delta_hrs'),
    wind_chill            = to_f32('wind_chill'),
    rush_rate_std_downs   = to_f32('rush_rate_std_downs_delta'),
    rush_rate_pass_downs  = to_f32('rush_rate_pass_downs_delta'),
    off_sack_rate_allowed = to_f32('off_sack_rate_allowed_delta'),
    def_sack_rate         = to_f32('def_sack_rate_delta'),

    off_archetype_idx = to_i32('off_archetype_idx'),
    def_archetype_idx = to_i32('def_archetype_idx'),

    last3_off_epa_avg      = to_f32('last3_off_epa_avg'),
    mask_last3_off_epa     = mask_last3_off_epa,
    last3_def_epa_avg      = to_f32('last3_def_epa_avg'),
    mask_last3_def_epa     = mask_last3_def_epa,
    last3_pts_scored_avg   = to_f32('last3_points_scored_avg'),
    mask_last3_pts_scored  = mask_last3_pts_scored,
    last3_pts_allowed_avg  = to_f32('last3_points_allowed_avg'),
    mask_last3_pts_allowed = mask_last3_pts_allowed,
    days_since_last_game   = to_f32('days_since_last_game'),
    mask_days_since        = mask_days_since,
    close_play_count_delta = to_f32('close_game_play_count_delta'),
    mask_close_play_count  = mask_close_play_count,
    away_elevation_delta   = to_f32('away_elevation_delta_ft'),
    mask_elevation         = mask_elevation,
)

# --- Verification ---
import dataclasses

N_rows = len(df)
for field in dataclasses.fields(training_data):
    val = getattr(training_data, field.name)
    if val is not None:
        assert val.shape[0] == N_rows, \
            f"{field.name} shape[0] = {val.shape[0]}, expected {N_rows}"

print(f"All {len(dataclasses.fields(training_data))} GameData fields verified: shape[0] == {N_rows}.")
print()

# Confirm archetype index dtypes and ranges
assert training_data.off_archetype_idx.dtype == jnp.int32, \
    f"off_archetype_idx dtype wrong: {training_data.off_archetype_idx.dtype}"
assert training_data.def_archetype_idx.dtype == jnp.int32, \
    f"def_archetype_idx dtype wrong: {training_data.def_archetype_idx.dtype}"
assert int(training_data.off_archetype_idx.min()) >= 0 and \
       int(training_data.off_archetype_idx.max()) <= 3, \
    f"off_archetype_idx out of range: [{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]"
assert int(training_data.def_archetype_idx.min()) >= 0 and \
       int(training_data.def_archetype_idx.max()) <= 3, \
    f"def_archetype_idx out of range: [{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]"

print(f"off_archetype_idx : dtype={training_data.off_archetype_idx.dtype}  "
      f"range=[{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]")
print(f"def_archetype_idx : dtype={training_data.def_archetype_idx.dtype}  "
      f"range=[{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]")
print()

# Confirm key continuous features are on unit scale (model_03 already scaled)
print("Scale check on continuous features (expect mean≈0, std≈1):")
check_cols = [
    ('sp_rating',               training_data.sp_rating),
    ('close_game_epa',          training_data.close_game_epa),
    ('pregame_elo',             training_data.pregame_elo),
    ('rush_rate_std_downs',     training_data.rush_rate_std_downs),
    ('last3_win_pct',           training_data.last3_win_pct),
]
for name, arr in check_cols:
    mean = float(arr.mean())
    std  = float(arr.std())
    print(f"  {name:<30s} mean={mean:+.4f}  std={std:.4f}")

print()
print("Cell 5 complete.")

All 35 GameData fields verified: shape[0] == 3214.

off_archetype_idx : dtype=int32  range=[0, 3]
def_archetype_idx : dtype=int32  range=[0, 3]

Scale check on continuous features (expect mean≈0, std≈1):
  sp_rating                      mean=-0.0000  std=0.9998
  close_game_epa                 mean=-0.0016  std=0.9914
  pregame_elo                    mean=-0.0000  std=0.9998
  rush_rate_std_downs            mean=-0.0000  std=0.9937
  last3_win_pct                  mean=-0.0000  std=0.9998

Cell 5 complete.


In [7]:
# =============================================================================
# CELL 6 — Run NUTS sampler (full production fit)
#
# Model definition identical to model_04 Cell 6.
# Only sampler settings change from diagnostic run:
#   num_warmup  : 200  -> 1000
#   num_samples : 200  -> 1000
#   num_chains  : 1    -> 4
#
# Expected wall-clock: 20-40 min on CPU depending on hardware.
# Do not interrupt. DB connection must remain open.
#
# Watch after completion:
#   sigma_attack : diagnostic mean=0.029 — confirm ESS_bulk/tail >= 400
#   hfa_league   : diagnostic mean=0.029 — confirm R-hat < 1.01
# =============================================================================

init_values = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "r_negbinom"         : jnp.ones(N_CONFERENCES) * 8.0,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_attack"       : jnp.array(0.05),
    "alpha_team_raw"     : jnp.zeros(N_teams),
    "sigma_defense"      : jnp.array(0.05),
    "delta_team_raw"     : jnp.zeros(N_teams),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

nuts_kernel = NUTS(
    model_cfb,
    target_accept_prob=0.9,
    init_strategy=init_to_value(values=init_values),
)

mcmc = MCMC(
    nuts_kernel,
    num_warmup=1000,
    num_samples=1000,
    num_chains=4,
    progress_bar=True,
)

print("Starting NUTS sampler (full production fit — 4 chains, 1000 warmup, 1000 samples)...")
print(f"  N_teams       : {N_teams}")
print(f"  N_games       : {len(df):,}")
print(f"  N_CONFERENCES : {N_CONFERENCES}")
print()

numpyro.enable_validation(False)
try:
    t0 = time.time()
    mcmc.run(random.PRNGKey(42), data=training_data, N_teams=N_teams)
    fit_time = time.time() - t0
finally:
    numpyro.enable_validation(True)

print(f"\nFit complete. Wall-clock time : {fit_time:.1f}s")

extra         = mcmc.get_extra_fields()
n_divergences = int(extra['diverging'].sum()) if 'diverging' in extra else 0
print(f"Divergences   : {n_divergences} / 4000")

samples = mcmc.get_samples()

# r_negbinom per-conference summary
r_vals = samples['r_negbinom']
print(f"\nr_negbinom shape : {r_vals.shape}  (expect (4000, {N_CONFERENCES}))")
print("r_negbinom posterior means by conference:")
for i, conf in enumerate(CONFERENCES):
    print(f"  {conf:<22s} mean={float(r_vals[:, i].mean()):.4f}  "
          f"std={float(r_vals[:, i].std()):.4f}")

print(f"\nmu_league    mean={float(samples['mu_league'].mean()):.4f}")
print(f"hfa_league   mean={float(samples['hfa_league'].mean()):.4f}")
print(f"sp_weight    mean={float(samples['sp_weight'].mean()):.4f}")
print(f"sigma_attack mean={float(samples['sigma_attack'].mean()):.4f}")
print(f"sigma_defense mean={float(samples['sigma_defense'].mean()):.4f}")

print(f"\nb_off_archetype shape : {samples['b_off_archetype'].shape}  (expect (4000, 4))")
print(f"b_def_archetype shape : {samples['b_def_archetype'].shape}  (expect (4000, 4))")
print()
print("Cell 6 complete.")

/var/folders/nd/7h3lcr8d2cjbxfmfqghczqz40000gn/T/ipykernel_14948/1137731784.py:62: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(


Starting NUTS sampler (full production fit — 4 chains, 1000 warmup, 1000 samples)...
  N_teams       : 131
  N_games       : 3,214
  N_CONFERENCES : 10



sample: 100%|████████████████████████████████████████████████████████████████| 2000/2000 [02:28<00:00, 13.45it/s, 127 steps of size 4.10e-02. acc. prob=0.91]



Fit complete. Wall-clock time : 648.6s
Divergences   : 0 / 4000

r_negbinom shape : (4000, 10)  (expect (4000, 10))
r_negbinom posterior means by conference:
  ACC                    mean=17.7713  std=1.9367
  American Athletic      mean=14.6956  std=1.7877
  Big 12                 mean=13.9734  std=1.5357
  Big Ten                mean=11.9773  std=1.3234
  Conference USA         mean=12.1143  std=1.4529
  Mid-American           mean=13.7708  std=1.6726
  Mountain West          mean=14.4431  std=1.8318
  Pac-12                 mean=15.8088  std=1.9141
  SEC                    mean=17.5407  std=1.9524
  Sun Belt               mean=14.5970  std=1.6040

mu_league    mean=3.1935
hfa_league   mean=0.0294
sp_weight    mean=0.0637
sigma_attack mean=0.0182
sigma_defense mean=0.0610

b_off_archetype shape : (4000, 4)  (expect (4000, 4))
b_def_archetype shape : (4000, 4)  (expect (4000, 4))

Cell 6 complete.


In [8]:
# =============================================================================
# CELL 7 — Convergence diagnostics and artifact save
#
# Required checks (session state thresholds):
#   R-hat < 1.01 for all parameters
#   ESS_bulk >= 400 for all parameters
#   ESS_tail >= 400 for all parameters
#
# Watch parameters flagged from diagnostic run:
#   sigma_attack : posterior mean 0.018 — near-zero HalfNormal; ESS priority
#   hfa_league   : posterior mean 0.029 — below prior center; R-hat priority
#
# Artifact saved to /Users/kevinjohnson/cfb-analytics/artifacts/
# =============================================================================

import arviz as az
import os
import shutil

ARTIFACTS_DIR = "/Users/kevinjohnson/cfb-analytics/artifacts"

# --- Convert to InferenceData ---
idata = az.from_numpyro(mcmc)

# --- R-hat and ESS summary ---
scalar_vars = [
    "mu_league", "hfa_league",
    "r_negbinom",
    "sigma_conference", "sigma_attack", "sigma_defense", "sigma_hfa_team",
    "sp_weight", "rec_weight_sunbelt",
    "b_close_game_epa", "b_close_game_def_epa",
    "b_pregame_elo", "b_elo_sp_divergence", "b_last3_win_pct",
    "b_away_travel_distance", "b_away_tz_delta", "b_wind_chill",
    "b_rush_rate_std_downs", "b_rush_rate_pass_downs",
    "b_off_sack_rate_allowed", "b_def_sack_rate",
    "b_off_archetype", "b_def_archetype",
    "b_last3_off_epa_avg", "b_last3_def_epa_avg",
    "b_last3_pts_scored_avg", "b_last3_pts_allowed_avg",
    "b_days_since_last_game", "b_close_play_count_delta",
    "b_away_elevation_delta",
]

summary = az.summary(idata, var_names=scalar_vars, round_to=4)
print("=== Convergence summary (scalar and low-dim parameters) ===")
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'ess_bulk', 'ess_tail', 'r_hat']].to_string())
print()

# --- Threshold checks ---
r_hat_vals        = summary['r_hat']
ess_bulk          = summary['ess_bulk']
ess_tail          = summary['ess_tail']

r_hat_failures    = r_hat_vals[r_hat_vals >= 1.01]
ess_bulk_failures = ess_bulk[ess_bulk < 400]
ess_tail_failures = ess_tail[ess_tail < 400]

print("=== Threshold check: R-hat < 1.01 ===")
if len(r_hat_failures) == 0:
    print("  PASS — all parameters R-hat < 1.01")
else:
    print(f"  FAIL — {len(r_hat_failures)} parameter(s) at or above threshold:")
    print(r_hat_failures.to_string())
print()

print("=== Threshold check: ESS_bulk >= 400 ===")
if len(ess_bulk_failures) == 0:
    print("  PASS — all parameters ESS_bulk >= 400")
else:
    print(f"  FAIL — {len(ess_bulk_failures)} parameter(s) below threshold:")
    print(ess_bulk_failures.to_string())
print()

print("=== Threshold check: ESS_tail >= 400 ===")
if len(ess_tail_failures) == 0:
    print("  PASS — all parameters ESS_tail >= 400")
else:
    print(f"  FAIL — {len(ess_tail_failures)} parameter(s) below threshold:")
    print(ess_tail_failures.to_string())
print()

# --- Per-team array diagnostics (max R-hat across 131 teams) ---
print("=== Team-level parameter diagnostics (max R-hat across 131 teams) ===")
for var in ["alpha_team_raw", "delta_team_raw", "hfa_team_raw"]:
    team_summary  = az.summary(idata, var_names=[var], round_to=4)
    max_rhat      = team_summary['r_hat'].max()
    min_ess_bulk  = team_summary['ess_bulk'].min()
    min_ess_tail  = team_summary['ess_tail'].min()
    rhat_status   = "PASS" if max_rhat < 1.01 else "FAIL"
    print(f"  {var:<20s}  max_rhat={max_rhat:.4f} [{rhat_status}]  "
          f"min_ess_bulk={min_ess_bulk:.0f}  min_ess_tail={min_ess_tail:.0f}")
print()

# --- Watch parameter detail ---
print("=== Watch parameter detail ===")
for var in ["sigma_attack", "hfa_league"]:
    row = summary.loc[var] if var in summary.index else None
    if row is not None:
        print(f"  {var:<20s}  mean={row['mean']:.4f}  "
              f"ess_bulk={row['ess_bulk']:.0f}  ess_tail={row['ess_tail']:.0f}  "
              f"r_hat={row['r_hat']:.4f}")
print()

# --- Artifact save ---
artifact_path = os.path.join(ARTIFACTS_DIR, "model_06_samples.pkl")
with open(artifact_path, "wb") as f:
    pickle.dump({
        "samples"      : samples,
        "idata"        : idata,
        "N_teams"      : N_teams,
        "N_CONFERENCES": N_CONFERENCES,
        "team_to_idx"  : team_to_idx,
        "conf_to_idx"  : conf_to_idx,
        "CONFERENCES"  : CONFERENCES,
    }, f)

assert os.path.exists(artifact_path), f"Save failed — file not found at {artifact_path}"
print(f"Artifact saved : {artifact_path}")
print(f"  Size         : {os.path.getsize(artifact_path) / 1e6:.1f} MB")
print(f"  samples keys : {sorted(samples.keys())}")

# --- Clean up misplaced artifacts from notebooks/Model/artifacts ---
old_dir = "/Users/kevinjohnson/cfb-analytics/notebooks/Model/artifacts"
for fname in ["model_06_samples.pkl", "model_05_prior_predictive.png",
              "scaler_stats.json", "archetype_label_maps.json"]:
    old_path = os.path.join(old_dir, fname)
    new_path = os.path.join(ARTIFACTS_DIR, fname)
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.move(old_path, new_path)
        print(f"Moved : {old_path} -> {new_path}")
    elif os.path.exists(old_path) and os.path.exists(new_path):
        os.remove(old_path)
        print(f"Removed duplicate : {old_path}")

print()
print("Cell 7 complete.")

=== Convergence summary (scalar and low-dim parameters) ===
                             mean      sd   hdi_3%  hdi_97%    ess_bulk   ess_tail   r_hat
mu_league                  3.1935  0.0964   3.0010   3.3648   2213.3608  2650.0486  1.0013
hfa_league                 0.0294  0.0124   0.0059   0.0520  10048.0632  2900.8707  0.9994
r_negbinom[0]             17.7713  1.9369  14.1797  21.4250   9750.0314  2911.8366  1.0015
r_negbinom[1]             14.6956  1.7879  11.3102  17.9152   9301.5252  2958.6819  1.0012
r_negbinom[2]             13.9734  1.5359  11.3389  17.0050   8281.9979  2759.0127  0.9998
r_negbinom[3]             11.9773  1.3236   9.5803  14.5174  10196.8988  3379.2787  1.0025
r_negbinom[4]             12.1143  1.4531   9.4065  14.8216   8694.0695  2840.1558  1.0021
r_negbinom[5]             13.7707  1.6728  10.6773  16.8350   8689.9228  2827.8380  1.0024
r_negbinom[6]             14.4431  1.8320  11.3083  18.0450   9275.1618  3426.8422  1.0021
r_negbinom[7]             15.8